## PakMart Traders – Silver to Gold

Joins Silver Delta tables and writes three pre-aggregated **Gold** tables for Power BI reporting.

| Gold table | Grain | Use |
|---|---|---|
| `aggregate_sale_by_date_city` | Date × City | Provincial & territory revenue |
| `aggregate_sale_by_date_employee` | Date × Salesperson | Sales rep performance |
| `aggregate_sale_by_date_category` | Date × City | Chiller vs. dry items by area |

> **Before running:** attach the `pakmart` lakehouse as default. Run **Bronze to Silver** first.

### Load Silver tables

In [ ]:
from pyspark.sql.functions import sum as _sum

fact_sale   = spark.table('fact_sale')
dim_date    = spark.table('dimension_date')
dim_city    = spark.table('dimension_city')
dim_emp     = spark.table('dimension_employee')

print('fact_sale rows  :', fact_sale.count())
print('dimension_date  :', dim_date.count())
print('dimension_city  :', dim_city.count())
print('dimension_emp   :', dim_emp.count())

### Gold 1 – Sale by Date & City

In [ ]:
sale_by_date_city = (
    fact_sale.alias('sale')
    .join(dim_date.alias('date'),
          fact_sale.InvoiceDateKey == dim_date.Date, 'inner')
    .join(dim_city.alias('city'),
          fact_sale.CityKey == dim_city.CityKey, 'inner')
    .select(
        'date.Date', 'date.CalendarMonthLabel', 'date.Day',
        'date.ShortMonth', 'date.CalendarYear',
        'date.FiscalYear', 'date.FiscalMonthLabel',
        'city.City', 'city.StateProvince', 'city.SalesTerritory',
        'sale.TotalExcludingTax', 'sale.TaxAmount',
        'sale.TotalIncludingTax', 'sale.Profit'
    )
    .groupBy(
        'date.Date', 'date.CalendarMonthLabel', 'date.Day',
        'date.ShortMonth', 'date.CalendarYear',
        'date.FiscalYear', 'date.FiscalMonthLabel',
        'city.City', 'city.StateProvince', 'city.SalesTerritory'
    )
    .agg(
        _sum('sale.TotalExcludingTax').alias('SumOfTotalExcludingTax'),
        _sum('sale.TaxAmount').alias('SumOfTaxAmount'),
        _sum('sale.TotalIncludingTax').alias('SumOfTotalIncludingTax'),
        _sum('sale.Profit').alias('SumOfProfit')
    )
    .orderBy('date.Date', 'city.StateProvince', 'city.City')
)

sale_by_date_city.write.mode('overwrite').format('delta') \
    .option('overwriteSchema', 'true') \
    .saveAsTable('aggregate_sale_by_date_city')

print('aggregate_sale_by_date_city:', spark.table('aggregate_sale_by_date_city').count(), 'rows')

### Gold 2 – Sale by Date & Salesperson

In [ ]:
sale_by_date_employee = (
    fact_sale.alias('sale')
    .join(dim_date.alias('date'),
          fact_sale.InvoiceDateKey == dim_date.Date, 'inner')
    .join(dim_emp.alias('emp'),
          fact_sale.SalespersonKey == dim_emp.EmployeeKey, 'inner')
    .select(
        'date.Date', 'date.CalendarMonthLabel', 'date.Day',
        'date.ShortMonth', 'date.CalendarYear',
        'date.FiscalYear', 'date.FiscalMonthLabel',
        'emp.PreferredName', 'emp.Employee',
        'sale.TotalExcludingTax', 'sale.TaxAmount',
        'sale.TotalIncludingTax', 'sale.Profit'
    )
    .groupBy(
        'date.Date', 'date.CalendarMonthLabel', 'date.Day',
        'date.ShortMonth', 'date.CalendarYear',
        'date.FiscalYear', 'date.FiscalMonthLabel',
        'emp.PreferredName', 'emp.Employee'
    )
    .agg(
        _sum('sale.TotalExcludingTax').alias('SumOfTotalExcludingTax'),
        _sum('sale.TaxAmount').alias('SumOfTaxAmount'),
        _sum('sale.TotalIncludingTax').alias('SumOfTotalIncludingTax'),
        _sum('sale.Profit').alias('SumOfProfit')
    )
    .orderBy('date.Date', 'emp.PreferredName')
)

sale_by_date_employee.write.mode('overwrite').format('delta') \
    .option('overwriteSchema', 'true') \
    .saveAsTable('aggregate_sale_by_date_employee')

print('aggregate_sale_by_date_employee:', spark.table('aggregate_sale_by_date_employee').count(), 'rows')

### Gold 3 – Sale by Date & City (chiller vs. dry items)

In [ ]:
from pyspark.sql.functions import sum as _sum

sale_by_date_category = (
    fact_sale.alias('sale')
    .join(dim_date.alias('date'),
          fact_sale.InvoiceDateKey == dim_date.Date, 'inner')
    .join(dim_city.alias('city'),
          fact_sale.CityKey == dim_city.CityKey, 'inner')
    .select(
        'date.Date', 'date.CalendarMonthLabel', 'date.CalendarYear',
        'date.FiscalYear', 'date.FiscalMonthLabel',
        'city.City', 'city.StateProvince', 'city.SalesTerritory',
        'sale.TotalExcludingTax', 'sale.TaxAmount',
        'sale.TotalIncludingTax', 'sale.Profit',
        'sale.TotalDryItems', 'sale.TotalChillerItems'
    )
    .groupBy(
        'date.Date', 'date.CalendarMonthLabel', 'date.CalendarYear',
        'date.FiscalYear', 'date.FiscalMonthLabel',
        'city.City', 'city.StateProvince', 'city.SalesTerritory'
    )
    .agg(
        _sum('sale.TotalExcludingTax').alias('SumOfTotalExcludingTax'),
        _sum('sale.TaxAmount').alias('SumOfTaxAmount'),
        _sum('sale.TotalIncludingTax').alias('SumOfTotalIncludingTax'),
        _sum('sale.Profit').alias('SumOfProfit'),
        _sum('sale.TotalDryItems').alias('TotalDryItems'),
        _sum('sale.TotalChillerItems').alias('TotalChillerItems')
    )
    .orderBy('date.Date', 'city.StateProvince')
)

sale_by_date_category.write.mode('overwrite').format('delta') \
    .option('overwriteSchema', 'true') \
    .saveAsTable('aggregate_sale_by_date_category')

print('aggregate_sale_by_date_category:', spark.table('aggregate_sale_by_date_category').count(), 'rows')

### Summary

In [ ]:
gold_tables = [
    'aggregate_sale_by_date_city',
    'aggregate_sale_by_date_employee',
    'aggregate_sale_by_date_category',
]
print(f'  {"Gold Table":<40}  {"Rows":>8}')
print('  ' + '-' * 50)
for t in gold_tables:
    print(f'  {t:<40}  {spark.table(t).count():>8,}')
print()
print('Silver → Gold complete.')